# B3 · Localización del compañero

**Spec:** [`docs/spec_B3_codex_target_localization.md`](../docs/spec_B3_codex_target_localization.md)  |  **Bloque:** B · Preparación  |  **Run de este set:** `ROXs12b_realigned`

Localiza el compañero de ROXs 12 B en el campo.

| | |
|---|---|
| **Entrada** | Cubo alineado |
| **Salida (QC/productos)** | `stages/stage01c_qc.json` |
| **Consume aguas abajo** | C1–C4 (posición de extracción) |


## Qué hace B3 y por qué

B3 **mide** las posiciones de la primaria y del compañero en el cubo de trabajo y las valida contra la astrometría publicada (Bowler+2017: sep ~1.78″, PA ~240°), sustituyendo las coordenadas *hardcodeadas* por coordenadas medidas. Escribe las coordenadas **canónicas** que TODAS las etapas siguientes (C1–C4, E4) leerán del QC — nunca más un número a mano.

**Decisiones clave:**
- **Banda de detección al ROJO (8800–9350 Å):** el compañero es un objeto subestelar **frío** — solo es detectable en el rojo. Colapsar ahí lo hace visible (SNR ~12).
- **Deriva cromática del centroide:** el centroide del compañero se desplaza ~3.56 px pico-a-pico con λ (>0.5 px umbral) → `chromatic_centroid_needed = True`; la extracción aguas abajo debe seguir el centroide dependiente de λ.
- La posición medida está a **5.23 px** de la vieja coordenada hardcodeada (72,152) → productos históricos pudieron usar la posición equivocada (open_issue).

El resultado (sep/PA que casan con Bowler+2017) confirma un **compañero real ligado**, insumo directo de la clasificación G4.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage01c_localize.sh --run-id $RUN
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage01c_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage01c_localize.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage01c_qc.json', RUN_ID)
nb.show(qc, keys=['companion.snr_detection', 'sep_arcsec', 'pa_deg', 'band_used_A', 'chromatic_centroid_needed'], title='B3')


## Resultados que llevaron a la conclusión

Posiciones, astrometría vs literatura y deriva cromática del `stage01c_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('B3', 'stages/stage01c_qc.json'):
        q = nb.load_qc('stages/stage01c_qc.json', RUN_ID)
        pr, cp, ast = q['primary'], q['companion'], q['astrometry']
        print(f"primaria  (y,x) = ({pr['pos_yx'][0]:.2f}, {pr['pos_yx'][1]:.2f})  ± {pr['err_px']:.2f} px")
        print(f"compañero (y,x) = ({cp['pos_yx'][0]:.2f}, {cp['pos_yx'][1]:.2f})  ± {cp['err_px']:.2f} px  |  SNR = {cp['snr_detection']:.1f}")
        print(f"banda de detección = {cp['band_used_A']} Å (rojo: compañero frío)")
        print()
        print(f"separación = {ast['sep_arcsec']:.3f}\" (esperado {ast['expected_sep_arcsec']}\", {ast['sep_deviation_sigma']:+.2f}σ)")
        print(f"PA         = {ast['pa_deg']:.2f}° (esperado {ast['expected_pa_deg']}°, {ast['pa_deviation_sigma']:+.2f}σ)")
        print()
        ch = q['chromatic']
        print(f"deriva cromática (compañero) = {ch['companion_drift_px_peak_to_peak']:.2f} px pico-a-pico "
              f"(umbral {ch['threshold_px']}) -> chromatic_centroid_needed = {ch['chromatic_centroid_needed']}")
        print(f"legacy hardcoded (x,y) = {q['legacy_check']['hardcoded_companion_xy']}  ->  medido a {q['legacy_check']['distance_px']:.2f} px")


## Plot — detección en la banda roja (primaria + compañero)

**FITS usado:** `stage02_xcorr_cube_stack.fits` (cubo de trabajo 170×170), colapsado en la banda de detección (8800–9350 Å). Marca la **primaria** (cyan), el **compañero** (círculo verde) y la **posición predicha** de Bowler+2017 (× blanca) — que cae sobre el compañero medido. El título lleva sep/PA medidos vs esperados.


In [ ]:
try:
    MAKE_PLOT = True   # cubo de trabajo (~0.4 GB); requiere kernel MUSE
    if MAKE_PLOT:
        try:
            import numpy as np
            import matplotlib.pyplot as plt
            from astropy.io import fits

            q = nb.load_qc('stages/stage01c_qc.json', RUN_ID)
            cp, pr, ast = q['companion'], q['primary'], q['astrometry']
            cube_path = q['input_cube']['file']
            print('FITS usado:', cube_path)
            from musepipe.io import read_wavelength_axis
            h = fits.open(cube_path, memmap=True); data = h[1].data
            data = np.asarray(data[0] if data.ndim == 4 else data, dtype=np.float32)
            wave = read_wavelength_axis(h, data_shape=data.shape)   # ext WAVELENGTH o WCS
            lo, hi = cp['band_used_A']; sel = (wave >= lo) & (wave <= hi)
            img = np.nanmedian(data[sel], axis=0); h.close()

            py, px = pr['pos_yx']; cy, cx = cp['pos_yx']; ppy, ppx = cp['predicted_pos_yx']
            v = np.nanpercentile(img, [5, 99.5])
            fig, ax = plt.subplots(figsize=(6.4, 6))
            ax.imshow(img, origin='lower', cmap='magma', vmin=v[0], vmax=v[1])
            ax.plot(px, py, '+', color='cyan', ms=14, mew=2, label=f'primaria ({px:.1f},{py:.1f})')
            ax.plot(cx, cy, 'o', mfc='none', mec='lime', ms=16, mew=2, label=f"compañero SNR={cp['snr_detection']:.1f}")
            ax.plot(ppx, ppy, 'x', color='white', ms=9, mew=1.5, label='predicho (Bowler+2017)')
            ax.set_title(f"B3 · banda {lo:.0f}-{hi:.0f} Å · sep={ast['sep_arcsec']:.3f}\" "
                         f"(esp {ast['expected_sep_arcsec']}) PA={ast['pa_deg']:.1f}° ({ast['pa_deviation_sigma']:+.2f}σ)")
            ax.legend(fontsize=8, loc='upper right'); ax.axis('off'); fig.tight_layout()
            outdir = nb.run_dir(RUN_ID) / 'plots' / 'b3_localize'; outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / 'detection.png', dpi=110); print('figura ->', outdir / 'detection.png'); plt.show()
        except Exception as e:
            print('No se pudo generar el plot:', type(e).__name__, e)
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## Decisiones y notas
- **Banda de detección al rojo (8800–9350 Å):** el compañero es una enana fría, solo detectable en el rojo (SNR ~12).
- **Astrometría casa con Bowler+2017 a <0.1σ** (sep 1.802″ vs 1.81, PA 240.1° vs 240) → compañero real ligado (insumo de G4).
- **Deriva cromática del centroide ~3.56 px** (>0.5 umbral) → `chromatic_centroid_needed`; C1–C4 deben seguir el centroide λ-dependiente.
- Coordenadas **canónicas desde QC** reemplazan la vieja hardcoded (72,152), que estaba a 5.23 px (productos históricos pudieron usar la posición equivocada).


## Conclusión (registrada)

**B3: compañero localizado en (y,x)=(155.6, 75.8), sep 1.802″ / PA 240.1°, casando con Bowler+2017 a <0.1σ; SNR 12.3.**

- **Fecha:** run realineado (cubo 2026-07-08).
- **Entrada:** `stage02_xcorr_cube_stack.fits`; **salida:** `stage01c_qc.json` + coords canónicas para C1–C4/E4.
- **Banda de detección:** 8800–9350 Å (rojo, compañero frío); primaria en (84.9, 84.6).
- **Deriva cromática:** 3.56 px pico-a-pico → `chromatic_centroid_needed = True`.
- **Legacy:** posición medida a 5.23 px de la hardcoded (72,152) — de ahí en adelante se usan las coords medidas.
- **Downstream:** C1–C4 y E4 leen la posición del compañero desde este QC.
